# NPRW Intan Analysis -- Jui-Ming Tsai

## Steps
Original .rhs file (Intan recorder)

    │
    ├─ Extract auxiliary data（stim pulse, IR beam, sync signals）
    │
    ├─ Neural signal readout（128ch, ~30kHz）
    │     │
    │     ├─ Channel rearrangement（device order → probe geometry）
    │     ├─ HPF（300 Hz）
    │     ├─ CLMR（ 30-150µm）
    │     └─ Artifact removal (zero-blanking)
    │           │
    │           └─ Save Prepocessing checkpoint
    │
    ├─── A: Threshold MUA ──────────────────┐
    │     ├─ MAD noise estimation                          
    │     ├─ Threshold Detection（3 MAD）           
    │     └─ Store peaks + metadata (.npz)          
    │                                               
    ├─── B: Kilosort 4 ─────────────────────┐       
    │     ├─ Concatenate 3 sessions       
    │     ├─ KS4 spike sorting (GPU)               
    │     ├─ SortingAnalyzer + Quality Metrics             
    │     ├─ SI GUI Manual curation                   
    │     └─ Threshold vs KS4 Comparison              
    │                                               
    └─────────── Further Progress ──────────────────────┘
            Add on : firing rate estimation
                      + trial alignment
                      → peri-event heatmap

## Control sessions
- **BR 002** → Baseline (Start), IR Trigger, No Stimulation
- **BR 003** → Baseline (Start), IR Trigger, No Stimulation
- **BR 032** → Baseline (End), IR Trigger, No Stimulation 

---
## 0. Imports + Config


In [1]:
import gc
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.io import loadmat

# SpikeInterface
from probeinterface import Probe
import spikeinterface as si
import spikeinterface.preprocessing as spre
import spikeinterface.extractors as se
import spikeinterface.sorters as ss
from spikeinterface.sortingcomponents.peak_detection import detect_peaks

# RCP pipeline
import RCP_analysis as rcp
from RCP_analysis.python.functions.config_loading import *

# Check env
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"SpikeInterface: {si.__version__}")
print(f"Session: {PARAMS.session}")

# Check Z-drive network mapping (Malfunction couple times so check these first)
if SESSION_LOC is None:
    print("\n[ERROR] SESSION_LOC mapping failed! Check Z: drive connection")
    _test_path = Path(PARAMS.data_root) / Path(PARAMS.location)

    print(f"  data_root: {PARAMS.data_root}")
    print(f"  location:  {PARAMS.location}")
    print(f"  Constructed path:  {_test_path}")
    print(f"  Path exists:  {_test_path.exists()}")

    if not _test_path.exists():
        print(f"\n  Z: drive inaccessible. Please check the following:")
        print(f"  1. Ensure vpn is connected and Z: drive is mapped correctly.")
        print(f"  2. Restart Jupyter kernel (Kernel → Restart)")
        print(f"  3. Or restart jupyter in the Anaconda Prompt:")
        print(f"     conda activate pipeline")
        print(f"     jupyter notebook")
    else:
        print(f"\n  Path exists but config_loading failed. Attempting manual setup...")
        SESSION_LOC = _test_path.resolve()
        OUT_BASE       = SESSION_LOC / "results"
        BR_ROOT        = SESSION_LOC / "Blackrock"
        VIDEO_ROOT     = SESSION_LOC / "Video"
        VOG_ROOT       = SESSION_LOC / "VOG"
        INTAN_ROOT     = SESSION_LOC / "Intan"
        METADATA_ROOT  = SESSION_LOC / "Metadata"
        BEHV_AUX_DATA  = OUT_BASE / "aux_data" / "Behavior"
        NPRW_AUX_DATA  = OUT_BASE / "aux_data" / "NPRW"
        UA_AUX_DATA    = OUT_BASE / "aux_data" / "UA"
        BEHV_CKPT_ROOT = OUT_BASE / "checkpoints" / "Behavior"
        NPRW_CKPT_ROOT = OUT_BASE / "checkpoints" / "NPRW"
        UA_CKPT_ROOT   = OUT_BASE / "checkpoints" / "UA"
        ALIGNED_CKPT_ROOT = OUT_BASE / "checkpoints" / "Aligned"
        VOG_CKPT_ROOT  = OUT_BASE / "checkpoints" / "VOG"
        PERI_ROOT      = OUT_BASE / "checkpoints" / "PeriStim"
        METADATA_CSV   = METADATA_ROOT / f"{PARAMS.session}_metadata.csv"
      
        for d in [OUT_BASE, NPRW_AUX_DATA, NPRW_CKPT_ROOT]:
            d.mkdir(parents=True, exist_ok=True)
        print(f"  Manual setup completed!")

print(f"\nSESSION_LOC: {SESSION_LOC}")
print(f"SESSION_LOC exists: {SESSION_LOC.exists() if SESSION_LOC else 'N/A'}")
print(f"INTAN_ROOT: {INTAN_ROOT}")
print(f"METADATA_CSV: {METADATA_CSV}")

CUDA available: True
SpikeInterface: 0.103.0
Session: NRR_RW012

SESSION_LOC: \\cullenstor.win.ad.jhu.edu\data\Current Project Databases - NHP\2025 Cerebellum prosthesis\Marvin\20260116_NRR_RW012
SESSION_LOC exists: True
INTAN_ROOT: \\cullenstor.win.ad.jhu.edu\data\Current Project Databases - NHP\2025 Cerebellum prosthesis\Marvin\20260116_NRR_RW012\Intan
METADATA_CSV: \\cullenstor.win.ad.jhu.edu\data\Current Project Databases - NHP\2025 Cerebellum prosthesis\Marvin\20260116_NRR_RW012\Metadata\NRR_RW012_metadata.csv


---
## 1. Probe Geometry

Objective: 
1. To read the physical coordinates of the probes (x/y position of each electrode contact) 
2. Create a Probe object for the ProbeInterface. Subsequent CLMR processes need to know which channels are "neighbors," depending on their physical distance from the probe.

In [2]:
# Load probe geometry matrix
GEOM_PATH = rcp.resolve_probe_geom_path(PARAMS, REPO_ROOT, session_key=None)
mat_probe = loadmat(str(GEOM_PATH))

intan_geom = {
    "x": mat_probe["xcoords"].ravel(), #.ravel() to convert from 2D (128,1) to 1D (128,)
    "y": mat_probe["ycoords"].ravel()
}
assert intan_geom["x"].size == intan_geom["y"].size, "x/y must have same length"

# chanMap0ind[i] 的意思是：「探針上第 i 個位置對應 Intan 記錄器的第幾個通道」
if "chanMap0ind" in mat_probe:
    intan_probe_mapping = mat_probe["chanMap0ind"].ravel()
else:
    raise ValueError("No 0-based chanmap in .mat geometry file.")

# ProbeInterface presentation
nprw_probe = Probe(ndim=2)
nprw_probe.set_contacts(
    positions=np.c_[intan_geom["x"], intan_geom["y"]], # 2 x (128,) --> (128, 2) shape
    shapes="square",
    shape_params={"width": 12.0} # 12 um width for square contacts
)
nprw_probe.set_device_channel_indices(intan_probe_mapping) # 將探針上的每個 contact 對應到 Intan 記錄器的 channel index

print(f"Probe: {intan_geom['x'].size} contacts")
print(f"Mapping shape: {intan_probe_mapping.shape}")
print(f"Y range: {intan_geom['y'].min():.0f} ~ {intan_geom['y'].max():.0f} um")

Probe: 128 contacts
Mapping shape: (128,)
Y range: 20 ~ 1280 um


---
## 2. Read Parameters

> from `params.yaml` 
- **neural**: 128ch neural signals (our primary data)

- **stim**: Electrical stimulation waveform recording (used to detect stim pulse timing)

- **aux**: Analog auxiliary channel (sync signal, heart rate, etc.)

- **ir**: Digital input channel (IR beam — triggered when a monkey reaches across)

- `RADII`: CLMR specifies the range of neighbor channels to monitor.

- `THRESH`: The spike detection threshold (3 => 3 times is considered a spike).

- `ARTRMV_MS`: The number of milliseconds to clear before and after stim artifacts.

In [3]:
# Configure hardware stream topologies
NPRW_CFG = PARAMS.probes.get("NPRW")
INTAN_STREAM = NPRW_CFG.get("neural_data_stream")   # "RHS2000 amplifier channel"
STIM_STREAM  = NPRW_CFG.get("stim_data_stream")     # "Stim channel"
AUX_STREAM   = NPRW_CFG.get("aux_stream")           # "USB board ADC input channel: "
IR_STREAM    = NPRW_CFG.get("ir_stream")            # "USB board digital input channel"

# CLMR Radius
# 距離在 30µm ~ 150µm 之間的鄰居通道（甜甜圈）then 計算這些鄰居的median，從當前ch減去
RADII = (
    NPRW_CFG.get("local_radius_inner"),  # 30 um
    NPRW_CFG.get("local_radius_outer"),  # 150 um
)

# Spike detection (MUA) parameters
RATES = PARAMS.NPRW_rate_est
BIN_MS     = RATES.get("bin_ms")           # 20 ms
SIGMA_MS   = RATES.get("sigma_ms")         # 20 ms
THRESH     = RATES.get("detect_threshold")  # 3
PEAK_SIGN  = RATES.get("peak_sign")         # negative, positive, or both
ARTRMV_MS_BEFORE = float(RATES.get("remove_ms_before", 20.0))
ARTRMV_TAIL_MS   = float(RATES.get("remove_tail_ms_after", 20.0))

# SpikeInterface parallel 
global_job_kwargs = dict(n_jobs=PARAMS.parallel_jobs, chunk_duration=PARAMS.chunk)
si.set_global_job_kwargs(**global_job_kwargs)

print(f"Streams: neural={INTAN_STREAM}, stim={STIM_STREAM}")
print(f"CLMR radii: inner={RADII[0]}, outer={RADII[1]} um")
print(f"Threshold: {THRESH}x noise, peak_sign={PEAK_SIGN}")
print(f"Artifact removal: {ARTRMV_MS_BEFORE} ms before, {ARTRMV_TAIL_MS} ms after")
print(f"Parallel: {PARAMS.parallel_jobs} jobs, chunk={PARAMS.chunk}")

Streams: neural=RHS2000 amplifier channel, stim=Stim channel
CLMR radii: inner=30, outer=150 um
Threshold: 3x noise, peak_sign=neg
Artifact removal: 20.0 ms before, 20.0 ms after
Parallel: 8 jobs, chunk=4s


---
## 3. Which Intan Sessions

> **BR 002, 003, 031**（control session）
> 這些沒有電刺激，是自然伸手的基線資料。

In [4]:
# All Intan session
all_sess = rcp.list_intan_sessions(INTAN_ROOT)
print(f"Total Intan sessions: {len(all_sess)}")
for s in all_sess[:5]:
    print(f"  {s.name}")
print(f"  ... ({len(all_sess)} total)")

# Metadata
meta_df = pd.read_csv(METADATA_CSV)
print(f"\nMetadata: {len(meta_df)} rows")
print(meta_df[['BR_File', 'Intan_File']].head(10))

Total Intan sessions: 31
  NRR_RW012_260116_131117
  NRR_RW012_260116_140816
  NRR_RW012_260116_141228
  NRR_RW012_260116_141652
  NRR_RW012_260116_141919
  ... (31 total)

Metadata: 29 rows
   BR_File  Intan_File
0        2           2
1        3           3
2        4           4
3        5           5
4        6           6
5        7           7
6        8           8
7        9           9
8       10          10
9       11          11


In [5]:
# Find BR 002, 003, 031 

# Compare（Both are int）
print("BR_File dtype:", meta_df['BR_File'].dtype)
print("BR_File values:", meta_df['BR_File'].values)
print("Intan_File values:", meta_df['Intan_File'].values)
print()

# BR_File 
CONTROL_BR_INDICES = [2, 3, 32]

control_meta = meta_df[meta_df['BR_File'].isin(CONTROL_BR_INDICES)]
print(f"Control sessions found: {len(control_meta)}")
print(control_meta[['BR_File', 'Intan_File']])
print()

# Match to Intan sessions
control_sessions = []
for _, row in control_meta.iterrows():
    intan_idx = int(row['Intan_File']) - 1  # turn 1-based Intan_File into 0-based index
    if 0 <= intan_idx < len(all_sess):
        sess_path = all_sess[intan_idx]
        control_sessions.append(sess_path)
        print(f"  BR {int(row['BR_File']):03d} → Intan #{int(row['Intan_File'])} → {sess_path.name}")
    else:
        print(f"  BR {int(row['BR_File']):03d} → Intan #{int(row['Intan_File'])} → INDEX OUT OF RANGE!")

print(f"\nMatched Intan folders: {len(control_sessions)}")
for s in control_sessions:
    print(f"  {s.name}  (exists: {s.exists()})")

BR_File dtype: int64
BR_File values: [ 2  3  4  5  6  7  8  9 10 11 12 13 14 15 17 18 19 20 21 22 23 24 25 27
 28 29 30 31 32]
Intan_File values: [ 2  3  4  5  6  7  8  9 10 11 12 13 14 15 17 18 19 20 21 22 23 24 25 26
 27 28 29 30 31]

Control sessions found: 3
    BR_File  Intan_File
0         2           2
1         3           3
28       32          31

  BR 002 → Intan #2 → NRR_RW012_260116_140816
  BR 003 → Intan #3 → NRR_RW012_260116_141228
  BR 032 → Intan #31 → NRR_RW012_260116_161621

Matched Intan folders: 3
  NRR_RW012_260116_140816  (exists: True)
  NRR_RW012_260116_141228  (exists: True)
  NRR_RW012_260116_161621  (exists: True)


---
## 4. Prepocessing

> From Bryan's NPRW_Intan_analysis_threshold.py


> Note 步驟：
> 1. 擷取輔助訊號（刺激波形、IR beam）
> 2. 讀取 128ch 神經資料
> 3. 高通濾波 300Hz
> 4. CLMR
> 5. Stim artifacrs去除（對control session 來說沒有）
> 6. 儲存結果

In [6]:
def preprocess_one_session(sess, nprw_probe, intan_probe_mapping):
    """
    Prepocessing 1 Intan session：
    HPF → CLMR → artifact removal → save
    
    return (rec_preprocessed, stim_ext_arrays, ir_ms, ir_idx)
    """
    print(f"\n{'='*60}")
    print(f"[START] Processing session: {sess.name}")
    print(f"{'='*60}")
    
    # --- 1) 擷取輔助訊號 ---
    print("[1/6] Extracting aux streams...")
    rcp.extract_intan_aux_streams_npz(
        sess=sess, out_dir=NPRW_AUX_DATA, aux_streams=AUX_STREAM
    )
    
    print("[2/6] Extracting stim streams...")
    stim_ext_arrays = rcp.extract_stim_npz(
        sess=sess, out_dir=NPRW_AUX_DATA,
        stim_stream_name=STIM_STREAM,
        chanmap_perm=intan_probe_mapping
    )
    
    # --- 2) 讀取 IR beam ---
    print("[3/6] Loading IR beam...")
    rec_ir = se.read_split_intan_files(
        sess, mode="concatenate",
        stream_name=IR_STREAM, use_names_as_ids=True
    )
    rec_ir = spre.unsigned_to_signed(rec_ir)
    fs_ir = float(rec_ir.sampling_frequency)
    sig = np.asarray(rec_ir.get_traces()).squeeze()
    ir_idx = rcp.detect_IR_crossings(sig, fs_ir, refractory_sec=0.0005)
    ir_ms = ir_idx / fs_ir * 1000.0
    print(f"  IR crossings: {ir_idx.size}")
    del rec_ir, sig; gc.collect()
    
    # --- 3) 讀取神經資料 + 重新排列通道 ---
    print("[4/6] Loading neural data + reordering...")
    rec = se.read_split_intan_files(
        sess, mode="concatenate",
        stream_name=INTAN_STREAM, use_names_as_ids=True
    )
    rec = spre.unsigned_to_signed(rec)
    rec = rec.set_probe(nprw_probe)
    rec_reordered = rcp.reorder_recording_to_geometry(rec, intan_probe_mapping)
    fs_nprw = rec_reordered.get_sampling_frequency()
    duration_s = rec.get_total_duration()
    print(f"  Channels: {rec_reordered.get_num_channels()}, fs: {fs_nprw} Hz, duration: {duration_s:.1f} s")
    
    # --- 4) HPF + CLMR ---
    print("[5/6] HPF 300Hz + CLMR...")
    rec_hp = spre.highpass_filter(rec_reordered, freq_min=float(PARAMS.highpass_hz))
    rec_ref = spre.common_reference(
        rec_hp, reference="local", operator="median",
        local_radius=(RADII[0], RADII[1])
    )
    
    # --- 5) 偽影去除 ---
    block_bounds = stim_ext_arrays.get("block_bounds_samples")
    stim_channels = stim_ext_arrays['active_channels']
    rec_artif_removed = rec_ref  # fallback if no stim
    n_total = rec_reordered.get_num_samples()
    dur_ms = None
    
    if block_bounds is not None and block_bounds.size:
        starts_samp = block_bounds[:, 0]
        ends_samp = block_bounds[:, 1]
        valid = (ends_samp > starts_samp) & (starts_samp >= 0) & (starts_samp < n_total)
        starts_samp = starts_samp[valid]
        ends_samp = ends_samp[valid]
        
        if starts_samp.size:
            dur_ms = (ends_samp - starts_samp) * 1000.0 / fs_nprw
            ms_after = float(dur_ms.max() + ARTRMV_TAIL_MS)
            rec_artif_removed = si.preprocessing.remove_artifacts(
                rec_ref,
                list_triggers=starts_samp.tolist(),
                ms_before=ARTRMV_MS_BEFORE,
                ms_after=ms_after,
                mode="zeros"
            )
            print(f"  Artifact removal: {starts_samp.size} stim blocks, blanking {ARTRMV_MS_BEFORE}+{ms_after:.1f} ms")
        else:
            print("  No valid stim blocks → skipping artifact removal (control session?)")
    else:
        print("  No stim detected → skipping artifact removal (control session)")
    
    # --- 6) 儲存前處理結果 ---
    print("[6/6] Saving preprocessed recording...")
    out_dir = NPRW_CKPT_ROOT / f"pp_local_{int(RADII[0])}_{int(RADII[1])}__interp_{sess.name}"
    rcp.save_recording(rec_artif_removed, out_dir)
    print(f"  Saved → {out_dir}")
    
    del rec_reordered, rec_hp, rec_ref; gc.collect()
    
    return rec_artif_removed, rec, stim_ext_arrays, ir_ms, ir_idx, dur_ms

---
## 5. Threshold MUA Detection

> From Bryan's NPRW_Intan_analysis_threshold.py

> Note: 對前處理過的 recording 用threshold 偵測 spike, 計算每個通道的雜訊 (MAD, 超過 3 倍的位置就是一個 spike。

In [7]:
def threshold_detect_and_save(rec_preprocessed, rec_raw, sess, ir_idx, ir_ms, stim_ext_arrays, dur_ms):

    fs_nprw = rec_preprocessed.get_sampling_frequency()
    stim_channels = stim_ext_arrays['active_channels']
    
    noise_levels = si.get_noise_levels(
        rec_preprocessed, method="mad", return_in_uV=False
    )
    print(f"  Noise levels: mean={np.nanmean(noise_levels):.2f}, std={np.nanstd(noise_levels):.2f}")
    
    peaks = detect_peaks(
        rec_preprocessed,
        method="by_channel_torch",
        detect_threshold=THRESH,
        peak_sign=PEAK_SIGN,
        noise_levels=noise_levels,
        n_jobs=1,  # Fix: Windows + PyTorch CUDA multiprocessing = BrokenProcessPool
    )
    print(f"  Peaks detected: {peaks.shape[0]}")
    
    out_npz = NPRW_CKPT_ROOT / f"rates__{sess.name}__bin{int(BIN_MS)}ms_sigma{int(SIGMA_MS)}ms.npz"
    save = dict(
        peaks=peaks,
        ir_idx=ir_idx,
        ir_ms=ir_ms,
        meta=dict(
            detect_threshold=THRESH,
            peak_sign=PEAK_SIGN,
            bin_ms=BIN_MS,
            sigma_ms=SIGMA_MS,
            fs=fs_nprw,
            stim_channels=stim_channels,
            stim_dur=dur_ms if dur_ms is not None else 0.0,
            n_channels=rec_preprocessed.get_num_channels(),
            session=str(sess.name),
            n_samples=rec_raw.get_total_samples(),
            n_segs=rec_raw.get_num_segments(),
            rec_dur=rec_raw.get_total_duration(),
            rec_start_ms=rec_raw.get_start_time() * 1000,
            rec_end_ms=rec_raw.get_end_time() * 1000,
        )
    )
    np.savez_compressed(out_npz, **save)
    print(f"  Saved → {out_npz}")
    return peaks

---
## 6. Run sessions（One by One）

> Perform complete preprocessing and threshold detection for each session.

In [8]:
# 想先測試一個就好: control_sessions[:1]

# Skip-if-done
_all_npz = []
_all_pp  = []
for _s in control_sessions:
    _all_npz.append(NPRW_CKPT_ROOT / f"rates__{_s.name}__bin{int(BIN_MS)}ms_sigma{int(SIGMA_MS)}ms.npz")
    _all_pp.append(NPRW_CKPT_ROOT / f"pp_local_{int(RADII[0])}_{int(RADII[1])}__interp_{_s.name}")

_npz_ok = all(p.exists() for p in _all_npz)
_pp_ok  = all(p.exists() for p in _all_pp)

if _npz_ok and _pp_ok:
    print("All Stage A results already on disk — skipping!")
    for p in _all_npz:
        d = np.load(str(p), allow_pickle=True)
        print(f"  {p.name}: {d['peaks'].shape[0]} peaks, {d['ir_idx'].shape[0]} IR crossings")
else:
    results = {}
    for sess in control_sessions:  
        # Preprocessing
        rec_pp, rec_raw, stim_arrays, ir_ms, ir_idx, dur_ms = preprocess_one_session(
            sess, nprw_probe, intan_probe_mapping
        )
        
        # Threshold MUA detection
        print(f"\n--- Threshold MUA detection ---")
        peaks = threshold_detect_and_save(
            rec_pp, rec_raw, sess, ir_idx, ir_ms, stim_arrays, dur_ms
        )
        
        results[sess.name] = {
            'n_peaks': peaks.shape[0],
            'n_ir': ir_idx.size,
            'duration_s': rec_raw.get_total_duration()
        }
        
        del rec_pp, rec_raw, peaks
        gc.collect()
        print(f"[DONE] {sess.name}")

    # Summary
    print(f"\n{'='*60}")
    print("Summary:")
    for name, r in results.items():
        print(f"  {name}: {r['n_peaks']} peaks, {r['n_ir']} IR crossings, {r['duration_s']:.1f}s")

All Stage A results already on disk — skipping!
  rates__NRR_RW012_260116_140816__bin20ms_sigma20ms.npz: 245385 peaks, 1 IR crossings
  rates__NRR_RW012_260116_141228__bin20ms_sigma20ms.npz: 1743101 peaks, 50 IR crossings
  rates__NRR_RW012_260116_161621__bin20ms_sigma20ms.npz: 3516627 peaks, 43 IR crossings


---
## Stage 2: Kilosort 4 Spike Sorting

> session concatenation --> sorting
>
> Note Steps：
> 1. 載入前處理 recording
> 2. concatenation
> 3. KS4 spike sorting
> 4. SortingAnalyzer 計算品質指標
> 5. SpikeInterface GUI 做視覺檢查 + 手動 curation

In [9]:
# 1. Load preprocessed results + concatenate recording

# check Kilosort 4 
import kilosort
print(f"Kilosort version: {kilosort.__version__}")
print(f"Available sorters: {[s for s in ss.available_sorters() if 'kilo' in s]}")
print()

# Load each control session's preprocessed recording
preprocessed_recs = []
session_boundaries = []  # Record the start points of each session in the concatenated recording (in sample numbers)
cumulative_samples = 0

for sess in control_sessions:
    pp_dir = NPRW_CKPT_ROOT / f"pp_local_{int(RADII[0])}_{int(RADII[1])}__interp_{sess.name}"
    if not pp_dir.exists():
        print(f"  [ERROR] {sess.name}: preprocessed data not found at {pp_dir}")
        print(f"  → Please run Preprocessing (Cell 6) first!")
        break
    rec = si.load_extractor(pp_dir)
    n_samples = rec.get_num_samples()
    preprocessed_recs.append(rec)
    session_boundaries.append({
        'name': sess.name,
        'start_sample': cumulative_samples,
        'end_sample': cumulative_samples + n_samples,
        'duration_s': rec.get_total_duration(),
    })
    cumulative_samples += n_samples
    print(f"  Loaded: {sess.name} — {rec.get_num_channels()}ch, {rec.get_total_duration():.1f}s, {n_samples} samples")

# concatenate to 1 recording
if len(preprocessed_recs) == len(control_sessions):
    rec_concat = si.concatenate_recordings(preprocessed_recs) 
    print(f"\nAfter concatenation:")
    print(f"  Channels: {rec_concat.get_num_channels()}")
    print(f"  Total duration: {rec_concat.get_total_duration():.1f}s")
    print(f"  Total samples: {rec_concat.get_num_samples()}")
    print(f"  Sampling rate: {rec_concat.get_sampling_frequency()} Hz")
    print(f"\nSession boundaries (for splitting spike times):")
    for sb in session_boundaries:
        print(f"  {sb['name']}: samples {sb['start_sample']:,} – {sb['end_sample']:,} ({sb['duration_s']:.1f}s)")
else:
    print("\n[ERROR] Cannot concatenate: Some session preprocessed data is missing!")

Kilosort version: 4.1.5
Available sorters: ['kilosort', 'kilosort2', 'kilosort2_5', 'kilosort3', 'kilosort4', 'pykilosort']



C:\Users\marvi\AppData\Local\Temp\ipykernel_36064\2675638693.py:20: DeprecationWarning: load_extractor() is deprecated and will be removed in version 0.104.0. Please use load() instead.
  rec = si.load_extractor(pp_dir)


  Loaded: NRR_RW012_260116_140816 — 128ch, 22.3s, 670464 samples
  Loaded: NRR_RW012_260116_141228 — 128ch, 166.1s, 4983552 samples
  Loaded: NRR_RW012_260116_161621 — 128ch, 296.4s, 8892160 samples

After concatenation:
  Channels: 128
  Total duration: 484.9s
  Total samples: 14546176
  Sampling rate: 30000.0 Hz

Session boundaries (for splitting spike times):
  NRR_RW012_260116_140816: samples 0 – 670,464 (22.3s)
  NRR_RW012_260116_141228: samples 670,464 – 5,654,016 (166.1s)
  NRR_RW012_260116_161621: samples 5,654,016 – 14,546,176 (296.4s)


In [10]:
# 2. Kilosort 4 Sorting (concatenated recording)

# ks4_output = NPRW_CKPT_ROOT / "kilosort4__concat_baselines"                 # ← original
ks4_output = NPRW_CKPT_ROOT / "kilosort4__concat_baselines_default"            # ← new

_ks4_done = (ks4_output / "sorter_output" / "params.py").exists()

if _ks4_done:
    print("KS4 results already on disk")
    sorting = ss.read_sorter_folder(str(ks4_output))
    print(f"  Units: {len(sorting.get_unit_ids())}")
    total_spikes = sum(sorting.get_unit_spike_train(u).size for u in sorting.get_unit_ids())
    print(f"  Total spikes: {total_spikes:,}")
else:
    print(f"KS4 output dir: {ks4_output}")
    print(f"Recording: {rec_concat.get_num_channels()}ch, {rec_concat.get_total_duration():.1f}s")
    print(f"Kilosort 4 Starting...")

    sorting = ss.run_sorter(
        sorter_name="kilosort4",
        recording=rec_concat,
        folder=str(ks4_output),
        verbose=True,
        remove_existing_folder=True,

        # KS4 parameters
        # do_CAR=False,                    # 已經做了 CLMR
        # skip_kilosort_preprocessing=True, # 跳過 KS4 內建的前處理
        # batch_size=60000,                # 每批次的樣本數
        # Th_universal=9,                  # spike 偵測門檻
        # Th_learned=8,                    # spike 偵測門檻（學習後）
        # do_correction=True,              # 漂移校正
        # nblocks=1,                       # 漂移校正分塊數（1 = 全域校正）
        torch_device="auto",             # 自動選擇 GPU
    )

    print(f"\nKilosort 4 Completed")
    print(f"  Units found: {len(sorting.get_unit_ids())}")
    total_spikes = sum(sorting.get_unit_spike_train(u).size for u in sorting.get_unit_ids())
    print(f"  Total spikes: {total_spikes:,}")

KS4 results already on disk
  Units: 120
  Total spikes: 666,375


In [11]:
# 3. Build SortingAnalyzer + metrics computation

from spikeinterface.curation import remove_excess_spikes

# analyzer_path = NPRW_CKPT_ROOT / "sorting_analyzer__concat_baselines"             
analyzer_path = NPRW_CKPT_ROOT / "sorting_analyzer__concat_baselines_default" 
_analyzer_done = (analyzer_path / "extensions").exists()

if _analyzer_done:
    print("SortingAnalyzer already on disk — loading!")
    analyzer = si.load_sorting_analyzer(str(analyzer_path))
    
    print(f"  Units: {len(analyzer.sorting.get_unit_ids())}")
    print(f"  Extensions: {list(analyzer.get_loaded_extension_names())}")
    qm_ext = analyzer.get_extension("quality_metrics")
    if qm_ext is not None:
        print(f"\nQuality Metrics (First 10 units):")
        print(qm_ext.get_data().head(10))
else:
    # 有些 spike time 超出 recording 總長，先修剪掉
    sorting_clean = remove_excess_spikes(sorting, rec_concat)
    n_removed = sum(sorting.get_unit_spike_train(u).size for u in sorting.get_unit_ids()) \
              - sum(sorting_clean.get_unit_spike_train(u).size for u in sorting_clean.get_unit_ids())
    print(f"Removed {n_removed} out-of-bounds spikes")

    print("Building SortingAnalyzer...")
    analyzer = si.create_sorting_analyzer(sorting_clean, rec_concat, sparse=True)

    # extensions
    print("Computing random_spikes + waveforms + templates...")
    analyzer.compute(["random_spikes", "waveforms", "templates"])

    print("Computing noise_levels + unit_locations...")
    analyzer.compute(["noise_levels", "unit_locations"])

    print("Computing spike_amplitudes...")
    analyzer.compute("spike_amplitudes")

    print("Computing correlograms + isi_histograms...")
    analyzer.compute(["correlograms", "isi_histograms"])

    print("Computing template_similarity...")
    analyzer.compute("template_similarity")

    print("Computing principal_components...")
    analyzer.compute("principal_components")

    print("Computing quality_metrics...")
    qm = analyzer.compute("quality_metrics")

    #  summary
    print(f"\n{'='*60}")
    print(f"SortingAnalyzer Completed")
    print(f"  Units: {len(sorting_clean.get_unit_ids())}")
    print(f"  Extensions computed: {list(analyzer.get_loaded_extension_names())}")
    print(f"\nQuality Metrics (First 10 units):")
    print(qm.get_data().head(10))

    # Save analyzer
    analyzer.save_as(folder=str(analyzer_path))
    print(f"\nAnalyzer saved → {analyzer_path}")

Removed 0 out-of-bounds spikes
Building SortingAnalyzer...


estimate_sparsity (workers: 8 processes):   0%|          | 0/122 [00:00<?, ?it/s]

Computing random_spikes + waveforms + templates...


compute_waveforms (workers: 8 processes):   0%|          | 0/122 [00:00<?, ?it/s]

Computing noise_levels + unit_locations...


...

c:\Users\marvi\anaconda3\envs\pipeline\lib\site-packages\spikeinterface\qualitymetrics\misc_metrics.py:1066: UserWarning: Some units have too few spikes : amplitude_cutoff is set to NaN
  warnings.warn(f"Some units have too few spikes : amplitude_cutoff is set to NaN")
c:\Users\marvi\anaconda3\envs\pipeline\lib\site-packages\numpy\core\_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\marvi\anaconda3\envs\pipeline\lib\site-packages\numpy\core\_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
c:\Users\marvi\anaconda3\envs\pipeline\lib\site-packages\numpy\core\_methods.py:198: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)


noise_level (workers: 8 processes):   0%|          | 0/20 [00:00<?, ?it/s]

calculate_pc_metrics:   0%|          | 0/120 [00:00<?, ?it/s]


SortingAnalyzer Completed
  Units: 120
  Extensions computed: ['random_spikes', 'waveforms', 'templates', 'noise_levels', 'unit_locations', 'spike_amplitudes', 'correlograms', 'isi_histograms', 'template_similarity', 'principal_components', 'quality_metrics']

Quality Metrics (First 10 units):
   num_spikes  firing_rate  presence_ratio       snr  isi_violations_ratio  \
0        4522     9.326162           0.625  0.722719              0.553278   
1        1607     3.314273           0.625  3.972710              0.250342   
2        3455     7.125584           0.625  5.707507              0.311414   
3        2627     5.417919           0.625  4.919210              0.117100   
4        1080     2.227390           0.750  4.211590              0.831400   
5        6305    13.003418           1.000  0.270425              0.256140   
6        3745     7.723679           1.000  3.428354              0.057620   
7       16869    34.790587           0.750  2.538653              0.595236   
8 

c:\Users\marvi\anaconda3\envs\pipeline\lib\site-packages\pandas\core\dtypes\cast.py:1060: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():
c:\Users\marvi\anaconda3\envs\pipeline\lib\site-packages\pandas\core\dtypes\cast.py:1084: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():


---
## Step 2-4：Spikeinterface GUI

> Or command：`sigui --mode=web --curation /path/to/analyzer`

In [12]:
# Save analyzer to local C: drive (fast, reliable)
local_save = Path(r"C:\Users\marvi\JHU_codespace\Cullen Lab\RCP_analysis\results\ks4_analyzer_default")
if (local_save / "extensions").exists():
    print("Already saved locally — loading!")
    analyzer = si.load_sorting_analyzer(str(local_save))
else:
    import shutil
    if local_save.exists():
        shutil.rmtree(local_save)  # remove empty folder
    saved = analyzer.save_as(folder=str(local_save), format="binary_folder")
    print(f"Saved: {local_save}")
    analyzer = saved
print(f"Extensions exist: {(local_save / 'extensions').exists()}")


Already saved locally — loading!
Extensions exist: True


In [13]:
# 4. SpikeInterface GUI — Load Analyzer + Launch
# Priority: local C: drive (fast, no VPN needed) → Z: drive fallback
from spikeinterface_gui import run_mainwindow
from pathlib import Path
import spikeinterface as si
from spikeinterface_gui import run_mainwindow

LOCAL_ANALYZER = Path(r"C:\Users\marvi\JHU_codespace\Cullen Lab\RCP_analysis\results\ks4_analyzer_default")
# ... rest of the cell stays the same


LOCAL_ANALYZER = Path(r"C:\Users\marvi\JHU_codespace\Cullen Lab\RCP_analysis\results\ks4_analyzer_default")
# LOCAL_ANALYZER = Path(r"C:\Users\marvi\JHU_codespace\Cullen Lab\RCP_analysis\results\test_analyzer")   # ← original

Z_ANALYZER = NPRW_CKPT_ROOT / "sorting_analyzer__concat_baselines_default"   
# Z_ANALYZER = NPRW_CKPT_ROOT / "sorting_analyzer__concat_baselines"    

if (LOCAL_ANALYZER / "extensions").exists():
    print(f"Loading analyzer from LOCAL: {LOCAL_ANALYZER}")
    analyzer = si.load_sorting_analyzer(str(LOCAL_ANALYZER))
elif (Z_ANALYZER / "extensions").exists():
    print(f"Loading analyzer from Z: drive: {Z_ANALYZER}")
    analyzer = si.load_sorting_analyzer(str(Z_ANALYZER))
else:
    raise FileNotFoundError("No saved analyzer found! Run cell 19 (B3) first.")

print(f"  Format: {analyzer.format}")
print(f"  Units: {len(analyzer.sorting.get_unit_ids())}")
print(f"\nLaunching GUI (desktop mode — Qt/OpenGL, faster than web)...")

run_mainwindow(analyzer, mode="web", curation=True)

Loading analyzer from LOCAL: C:\Users\marvi\JHU_codespace\Cullen Lab\RCP_analysis\results\ks4_analyzer_default
  Format: binary_folder
  Units: 120

Launching GUI (desktop mode — Qt/OpenGL, faster than web)...


c:\Users\marvi\anaconda3\envs\pipeline\lib\site-packages\spikeinterface_gui\utils_panel.py:12: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension("tabulator")


c:\Users\marvi\anaconda3\envs\pipeline\lib\site-packages\spikeinterface_gui\unitlistview.py:449: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension("tabulator")


c:\Users\marvi\anaconda3\envs\pipeline\lib\site-packages\spikeinterface_gui\mergeview.py:312: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension("tabulator")


c:\Users\marvi\anaconda3\envs\pipeline\lib\site-packages\spikeinterface_gui\curationview.py:300: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension("tabulator")


c:\Users\marvi\anaconda3\envs\pipeline\lib\site-packages\spikeinterface_gui\backend_panel.py:278: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension("gridstack")


Found available port: 50948
Launching server at http://localhost:50948


---
## Step 2-5. Results and Comparison

> load curation.json and compare threshold vs KS4。

In [15]:
# check curation result
import matplotlib.pyplot as plt

curation_json = analyzer_path / "spikeinterface_gui_curation.json"
if curation_json.exists():
    from spikeinterface.curation import load_curation, apply_curation
    curation = load_curation(str(curation_json))
    curated_sorting = apply_curation(analyzer, curation_dict_or_model=curation)
    good_units = curated_sorting.get_unit_ids()
    print(f"Curation loaded: {len(good_units)} units after curation")
else:
    print("No curation file found — using raw KS4 sorting results")
    good_units = sorting.get_unit_ids()

# KS4 summary after curation
ks4_spike_counts = []
for uid in good_units:
    n = sorting.get_unit_spike_train(uid).size
    ks4_spike_counts.append((uid, n))
ks4_spike_counts.sort(key=lambda x: -x[1])
ks4_total = sum(n for _, n in ks4_spike_counts)

print(f"\nKS4 Results:")
print(f"  Units: {len(good_units)}")
print(f"  Total spikes: {ks4_total:,}")
print(f"  Top 10 units by spike count:")
for uid, n in ks4_spike_counts[:10]:
    print(f"    Unit {uid}: {n:,} spikes")

# compare to threshold results
thresh_totals = {}
for sess in control_sessions:
    npz_path = NPRW_CKPT_ROOT / f"rates__{sess.name}__bin{int(BIN_MS)}ms_sigma{int(SIGMA_MS)}ms.npz"
    if npz_path.exists():
        data = np.load(str(npz_path), allow_pickle=True)
        thresh_totals[sess.name] = data['peaks'].shape[0]
    else:
        thresh_totals[sess.name] = 0
thresh_total = sum(thresh_totals.values())
print(f"\nThreshold MUA Results:")
for name, n in thresh_totals.items():
    print(f"  {name}: {n:,} peaks")
print(f"  Total: {thresh_total:,}")

# plot comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: overall comparison
ax = axes[0]
ax.bar(['Threshold\n(MUA)', 'Kilosort 4\n(sorted)'],
       [thresh_total, ks4_total],
       color=['steelblue', 'coral'])
ax.set_ylabel('Total spike count')
ax.set_title('Spike Detection: Threshold vs KS4')

# Center: KS4 unit spike distribution
ax = axes[1]
counts = [n for _, n in ks4_spike_counts]
ax.hist(counts, bins=30, color='coral', edgecolor='black', alpha=0.7)
ax.set_xlabel('Spikes per unit')
ax.set_ylabel('Number of units')
ax.set_title(f'KS4 Unit Distribution ({len(good_units)} units)')

# Right: per-session threshold
ax = axes[2]
names = [Path(n).name[:20] for n in thresh_totals.keys()]
ax.bar(names, thresh_totals.values(), color='steelblue')
ax.set_ylabel('Peak count')
ax.set_title('Threshold Peaks per Session')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

No curation file found — using raw KS4 sorting results

KS4 Results:
  Units: 120
  Total spikes: 666,375
  Top 10 units by spike count:
    Unit 84: 75,350 spikes
    Unit 16: 37,284 spikes
    Unit 83: 25,391 spikes
    Unit 85: 25,103 spikes
    Unit 69: 20,927 spikes
    Unit 7: 16,869 spikes
    Unit 51: 16,254 spikes
    Unit 81: 15,604 spikes
    Unit 28: 15,268 spikes
    Unit 76: 15,034 spikes

Threshold MUA Results:
  NRR_RW012_260116_140816: 245,385 peaks
  NRR_RW012_260116_141228: 1,743,101 peaks
  NRR_RW012_260116_161621: 3,516,627 peaks
  Total: 5,505,113


C:\Users\marvi\AppData\Local\Temp\ipykernel_36064\707034980.py:73: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Next Steps

1. **Show Bryan results:** KS4 units, quality metrics, waveform templates
2. **Bryan will teach:** firing rate estimation + trial alignment → peri-event heatmap
3. **Goal:** z-score neuron x time heatmap
4. **Later:** expand to stim sessions, compare baseline vs stimulation

### Reload analyzer:
\```python
# Local (no VPN needed):
analyzer = si.load_sorting_analyzer(r"C:\Users\marvi\...\results\test_analyzer")
# Z: drive:
analyzer = si.load_sorting_analyzer(str(NPRW_CKPT_ROOT / "sorting_analyzer__concat_baselines"))
\```

### CLI GUI launch (doesn't block Jupyter kernel):
\```bash
conda activate pipeline
sigui --mode=web --curation "C:\Users\marvi\JHU_codespace\Cullen Lab\RCP_analysis\results\test_analyzer"
\```

### Session boundaries (split spike times back to individual sessions):
\```python
for sb in session_boundaries:
    print(f"{sb['name']}: samples {sb['start_sample']} – {sb['end_sample']}")
\```

### Curation guide:
See `notebooks/curation_guide_zh-tw.md` for detailed instructions.
